# 01: Pilot data inspection and shortcut check (E0, E0b)

Looks at a small pilot set of real iPhone photos and ChatGPT images. It checks whether simple file properties (format, size, metadata) can separate real from fake **without looking at pixels**, normalizes the files, and checks again.

Expected Drive layout: `MyDrive/minor-project/data/{real,fake}/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("GPU available:", torch.cuda.is_available())

ROOT = "/content/drive/MyDrive/minor-project"

In [ ]:
!pip install -q pillow-heif

## E0: Inspect raw files

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ExifTags, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".heic", ".heif"}


def describe(path):
    img = Image.open(path)
    arr = np.array(img)
    exif = img.getexif()

    camera = f"{exif.get(271, '')} {exif.get(272, '')}".strip() or "none"
    software = exif.get(305, "none")

    print(f"Format: {img.format} | Mode: {img.mode} | Size (W x H): {img.size[0]} x {img.size[1]}")
    print(f"As numbers: shape {arr.shape}, type {arr.dtype}, values from {arr.min()} to {arr.max()}")
    print(f"EXIF tags: {len(exif)} | Camera: {camera} | Software: {software}")
    print(f"Hidden extra info: {list(img.info.keys())}")

    return {
        "file": path.name,
        "format": img.format,
        "mode": img.mode,
        "width": img.size[0],
        "height": img.size[1],
        "file_kb": round(path.stat().st_size / 1024),
        "exif_tags": len(exif),
        "camera": camera,
        "hidden_info": len(img.info),
    }


def inspect_folder(base):
    rows = []
    for label in ["real", "fake"]:
        print(f"\n==================== {label.upper()} ====================")
        for path in sorted(Path(base, label).iterdir()):
            if path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            print(f"\n--- {path.name} ---")
            info = describe(path)
            info["label"] = label
            rows.append(info)
    return pd.DataFrame(rows)


Path(ROOT, "results").mkdir(exist_ok=True)
summary = inspect_folder(Path(ROOT, "data"))
summary.to_csv(Path(ROOT, "results", "01_data_inspection.csv"), index=False)
summary

In [ ]:
RULES = {
    "file format is PNG": lambda t: t["format"] == "PNG",
    "width under 1000 pixels": lambda t: t["width"] < 1000,
    "file smaller than 1 MB": lambda t: t["file_kb"] < 1024,
    "no EXIF tags": lambda t: t["exif_tags"] == 0,
    "no hidden info": lambda t: t["hidden_info"] == 0,
}


def run_cheat_rules(table):
    for name, rule in RULES.items():
        prediction = np.where(rule(table), "fake", "real")
        accuracy = (prediction == table["label"]).mean()
        print(f"Cheat rule '{name}': accuracy = {accuracy:.0%}")


run_cheat_rules(summary)

## E0b: Normalize (rotate upright, convert to RGB, center crop to 512, rebuild from pixels, save as PNG)

In [ ]:
SIZE = 512
CLEAN = Path(ROOT, "data_clean")


def normalize(path, size=SIZE):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")

    width, height = img.size
    if width < size or height < size:
        return None

    left = (width - size) // 2
    top = (height - size) // 2
    img = img.crop((left, top, left + size, top + size))
    return Image.fromarray(np.array(img))


for label in ["real", "fake"]:
    (CLEAN / label).mkdir(parents=True, exist_ok=True)
    for path in sorted(Path(ROOT, "data", label).iterdir()):
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        clean = normalize(path)
        if clean is None:
            print(f"Skipped {path.name}: smaller than {SIZE} pixels")
            continue
        out_path = CLEAN / label / f"{path.stem}.png"
        clean.save(out_path)
        print(f"Saved {label}/{out_path.name}")

In [ ]:
clean_summary = inspect_folder(CLEAN)
clean_summary.to_csv(Path(ROOT, "results", "02_clean_inspection.csv"), index=False)
run_cheat_rules(clean_summary)

In [ ]:
import matplotlib.pyplot as plt

paths = sorted(Path(CLEAN, "real").glob("*.png")) + sorted(Path(CLEAN, "fake").glob("*.png"))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, path in zip(axes.flat, paths):
    ax.imshow(Image.open(path))
    ax.set_title(f"{path.parent.name}: {path.stem[:15]}", fontsize=9)
    ax.axis("off")
for ax in axes.flat[len(paths):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(Path(ROOT, "results", "02_clean_images.png"), dpi=100)
plt.show()